# Colab — End-to-End Fine-Tune & Predict
**Multilingual Health QA (HASH / Zindi).** Reproducible run: install → train seq2seq →
evaluate on Val (leaderboard-mirroring ROUGE) → generate a **reviewable** Test prediction
CSV. Maps to rubric crit. 1, 2, 9 (leaderboard, tracking, reproducibility).

> ⚠️ This notebook does **not** upload anything. It writes a local submission CSV for you
> to review and submit manually.

## 0. Runtime check (use a GPU runtime: Runtime → Change runtime type → T4/A100)

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0),
          f'{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## 1. Get the code + data

In [ ]:
import sys, os
if 'google.colab' in sys.modules:
    !git clone -q https://github.com/NzizaPacifique250/multilingual_health_project.git
    %cd multilingual_health_project
    !pip install -q -r requirements.txt
sys.path.insert(0, os.getcwd())
# Data must be in ./datasets (Train.csv, Val.csv, Test.csv, SampleSubmission.csv).
# On Colab, upload them or mount Drive if not tracked in the repo.
assert os.path.exists('datasets/Train.csv'), 'place data CSVs in datasets/'

## 2. Train — run **B2** (mt5-base + LoRA, improved over B1)
B1 (mt5-base LoRA, 3 epochs) scored proxy **0.181**, below the TF-IDF floor (0.394) — it
underfit and under-generated, with Lug_Uga (0.096) and Eng_Uga (0.124) weakest. **B2** keeps
the same model + LoRA config and changes only data + decoding + training length:
- `--epochs 5` (more LoRA steps — B1 underfit)
- `--upsample_low_resource 3.0` (repeat Amh/Swa/Lug/Aka ×3)
- `--num_beams 5 --length_penalty 1.3 --gen_min_len 16` (recall-oriented decoding)

> ⚠️ **Use an A100 or L4 runtime if possible.** mT5 is numerically unstable in **fp16** (T4)
> and can diverge to `NaN`/`inf` loss; A100/L4 support **bf16**, which is stable. The cell
> below auto-selects bf16 when available and falls back to fp16 otherwise.

In [ ]:
import torch
# mT5 + fp16 can diverge to NaN; prefer bf16 (A100/L4). Auto-select.
PREC = "--bf16" if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else "--fp16"
print("precision:", PREC, "| device:",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# B2: mt5-base + LoRA, low-resource upsampling x3, recall-oriented decoding, 5 epochs.
# ~3-5h on T4 / ~1.5-2h on A100. LoRA + gradient checkpointing keep it in <=16GB VRAM.
!python -m src.train \
    --model_name google/mt5-base \
    --output_dir outputs/B2_mt5base_lora \
    --use_lora --lora_r 16 --lora_alpha 32 --lora_dropout 0.05 \
    --epochs 5 --train_bs 8 --eval_bs 8 --grad_accum 2 \
    --lr 3e-4 --max_input_len 128 --max_target_len 256 \
    --upsample_low_resource 3.0 \
    --num_beams 5 --gen_max_len 256 --gen_min_len 16 \
    --no_repeat_ngram 3 --length_penalty 1.3 \
    --eval_subsample 1500 --gradient_checkpointing {PREC}

## 3. Evaluate on Val (full per-language ROUGE)

In [ ]:
!python -m src.infer --model_dir outputs/B2_mt5base_lora --split val \
    --num_beams 5 --gen_max_len 256 --gen_min_len 16 \
    --no_repeat_ngram 3 --length_penalty 1.3 --tag B2_mt5base_lora

## 4. Generate Test predictions → reviewable submission CSV (no upload)

In [ ]:
!python -m src.infer --model_dir outputs/B2_mt5base_lora --split test \
    --num_beams 5 --gen_max_len 256 --gen_min_len 16 \
    --no_repeat_ngram 3 --length_penalty 1.3 --tag B2_mt5base_lora

In [ ]:
import pandas as pd
sub = pd.read_csv('outputs/submission_B2_mt5base_lora.csv')
print(sub.shape); sub.head()  # REVIEW before uploading to Zindi

## 5. Manual review checklist (do this before submitting)
- [ ] Spot-check Amharic rows are **Ge'ez script** (infer.py prints a wrong-language count).
- [ ] No empty / single-token answers; lengths look reasonable per language.
- [ ] Val proxy **beats the TF-IDF B0 (0.394)** and prior runs (`experiments/results.csv`).
- [ ] IDs match `SampleSubmission.csv` (submit.py validates this).
Then upload `outputs/submission_*.csv` to Zindi yourself and record the LB score in the log.

## 6. Log the run
Append a row to `experiments/results.csv` and a note to `experiments/log.md`:
config (model, epochs, lr, decoding), Val R1/RL/proxy, LB score, and one-line takeaway.